In [28]:
import pandas as pd
transactions = pd.read_parquet("../../data/processed/transactions_clean.parquet")
crm = pd.read_parquet("../../data/generated/crm_customers.parquet")
marketing = pd.read_parquet("../../data/generated/marketing_preferences.parquet")
loyalty = pd.read_parquet("../../data/generated/loyalty_program.parquet")
support = pd.read_parquet("../../data/generated/support_tickets.parquet")
returns = pd.read_parquet("../../data/generated/returns.parquet")
payments = pd.read_parquet("../../data/generated/payments.parquet")
campaigns = pd.read_parquet("../../data/generated/campaigns.parquet")
sends = pd.read_parquet("../../data/generated/campaign_sends.parquet")

sources = {
    "crm": crm, "marketing": marketing, "loyalty": loyalty,
    "support": support, "returns": returns, "payments": payments,
    "campaign_sends": sends,
}

expected = {
    "crm": 5494, "marketing": 5334, "loyalty": 2958,
    "support": 1809, "returns": 43467, "payments": 33872,
    "campaign_sends": 25852,
}

for name, df in sources.items():
    status = "OK" if len(df) == expected[name] else "MISMATCH"
    print(f"{name:15s} attendu={expected[name]:>8} obtenu={len(df):>8}  [{status}]")

print("Nb campagnes :", len(campaigns), "(attendu 12)")
print("Plage signup_date CRM :", crm["signup_date"].min(), "->", crm["signup_date"].max())

crm             attendu=    5494 obtenu=    5494  [OK]
marketing       attendu=    5334 obtenu=    5334  [OK]
loyalty         attendu=    2958 obtenu=    2958  [OK]
support         attendu=    1809 obtenu=    1809  [OK]
returns         attendu=   43467 obtenu=   43467  [OK]
payments        attendu=   33872 obtenu=   33872  [OK]
campaign_sends  attendu=   25852 obtenu=   25852  [OK]
Nb campagnes : 12 (attendu 12)
Plage signup_date CRM : 2008-01-01 00:00:00 -> 2011-10-02 00:00:00


In [29]:
from pathlib import Path

paths = {
    "transactions": "../../data/processed/transactions_clean.parquet",
    "crm": "../../data/generated/crm_customers.parquet",
    "marketing": "../../data/generated/marketing_preferences.parquet",
    "loyalty": "../../data/generated/loyalty_program.parquet",
    "support": "../../data/generated/support_tickets.parquet",
    "returns": "../../data/generated/returns.parquet",
    "payments": "../../data/generated/payments.parquet",
    "campaigns": "../../data/generated/campaigns.parquet",
    "campaign_sends": "../../data/generated/campaign_sends.parquet",
}

for name, rel_path in paths.items():
    p = Path(rel_path).resolve()  # transforme le chemin relatif en chemin absolu réel
    print(f"{name:15s} existe={p.exists()!s:5}  ->  {p}")

transactions    existe=True   ->  C:\Users\Moi\OneDrive\Bureau\customer-analytics-platform\data\processed\transactions_clean.parquet
crm             existe=True   ->  C:\Users\Moi\OneDrive\Bureau\customer-analytics-platform\data\generated\crm_customers.parquet
marketing       existe=True   ->  C:\Users\Moi\OneDrive\Bureau\customer-analytics-platform\data\generated\marketing_preferences.parquet
loyalty         existe=True   ->  C:\Users\Moi\OneDrive\Bureau\customer-analytics-platform\data\generated\loyalty_program.parquet
support         existe=True   ->  C:\Users\Moi\OneDrive\Bureau\customer-analytics-platform\data\generated\support_tickets.parquet
returns         existe=True   ->  C:\Users\Moi\OneDrive\Bureau\customer-analytics-platform\data\generated\returns.parquet
payments        existe=True   ->  C:\Users\Moi\OneDrive\Bureau\customer-analytics-platform\data\generated\payments.parquet
campaigns       existe=True   ->  C:\Users\Moi\OneDrive\Bureau\customer-analytics-platform\data\ge

In [30]:
# Inventaire de qualité systématique : nulls, doublons, taille
def quality_report(name, df, key="customer_id"):
    print(f"=== {name} ===")
    print("Lignes :", len(df))
    if key in df.columns:
        print(f"Doublons sur {key} :", df[key].duplicated().sum())
    nulls = df.isna().sum()
    nulls = nulls[nulls > 0]
    if len(nulls):
        print("Valeurs manquantes :")
        print(nulls)
    print()

for name, df in sources.items():
    quality_report(name, df)

=== crm ===
Lignes : 5494
Doublons sur customer_id : 160
Valeurs manquantes :
email    213
dtype: int64

=== marketing ===
Lignes : 5334
Doublons sur customer_id : 0
Valeurs manquantes :
opted_in_date    292
dtype: int64

=== loyalty ===
Lignes : 2958
Doublons sur customer_id : 0
Valeurs manquantes :
enrollment_date    161
dtype: int64

=== support ===
Lignes : 1809
Doublons sur customer_id : 476
Valeurs manquantes :
satisfaction_score    612
dtype: int64

=== returns ===
Lignes : 43467
Doublons sur customer_id : 39064

=== payments ===
Lignes : 33872
Doublons sur customer_id : 28538

=== campaign_sends ===
Lignes : 25852
Doublons sur customer_id : 20273
Valeurs manquantes :
opened       8619
clicked      8619
converted    8619
dtype: int64



In [23]:
print(crm.columns)
print(marketing.columns)
print(loyalty.columns)
print(support.columns)
print(returns.columns)
print(payments.columns)
print(campaigns.columns)
print(sends.columns)

Index(['customer_id', 'first_name', 'last_name', 'email', 'phone', 'country',
       'signup_date'],
      dtype='object')
Index(['customer_id', 'preferred_channel', 'consent_email', 'consent_sms',
       'opted_in_date'],
      dtype='object')
Index(['customer_id', 'loyalty_id', 'tier', 'points_balance',
       'enrollment_date'],
      dtype='object')
Index(['ticket_id', 'customer_id', 'category', 'opened_date', 'resolved_date',
       'satisfaction_score'],
      dtype='object')
Index(['return_id', 'customer_id', 'invoice_id', 'stock_code', 'purchase_date',
       'return_date', 'refund_amount'],
      dtype='object')
Index(['payment_id', 'invoice_id', 'customer_id', 'payment_date', 'amount',
       'method', 'status'],
      dtype='object')
Index(['campaign_id', 'campaign_type', 'sent_date'], dtype='object')
Index(['send_id', 'campaign_id', 'customer_id', 'sent_date', 'opened',
       'clicked', 'converted'],
      dtype='object')


In [31]:
# Vérification corrigée : la bonne clé dépend du grain de chaque table
quality_report("crm", crm, key="customer_id")           # grain client -> OK tel quel
quality_report("marketing", marketing, key="customer_id")
quality_report("loyalty", loyalty, key="customer_id")

quality_report("support", support, key="ticket_id")             # grain événement
quality_report("returns", returns, key="return_id")
quality_report("payments", payments, key="payment_id")
quality_report("campaign_sends", sends, key="send_id")

=== crm ===
Lignes : 5494
Doublons sur customer_id : 160
Valeurs manquantes :
email    213
dtype: int64

=== marketing ===
Lignes : 5334
Doublons sur customer_id : 0
Valeurs manquantes :
opted_in_date    292
dtype: int64

=== loyalty ===
Lignes : 2958
Doublons sur customer_id : 0
Valeurs manquantes :
enrollment_date    161
dtype: int64

=== support ===
Lignes : 1809
Doublons sur ticket_id : 0
Valeurs manquantes :
satisfaction_score    612
dtype: int64

=== returns ===
Lignes : 43467
Doublons sur return_id : 0

=== payments ===
Lignes : 33872
Doublons sur payment_id : 0

=== campaign_sends ===
Lignes : 25852
Doublons sur send_id : 256
Valeurs manquantes :
opened       8619
clicked      8619
converted    8619
dtype: int64



In [32]:
real_customer_ids = set(transactions["Customer ID"].unique())

for name, df in {"loyalty": loyalty, "campaign_sends": sends}.items():
    orphans = (~df["customer_id"].isin(real_customer_ids)).sum()
    print(f"{name:15s} clients orphelins : {orphans}")

loyalty         clients orphelins : 25
campaign_sends  clients orphelins : 256


In [20]:
dup_invoices = payments["invoice_id"].duplicated().sum()
print("Factures payées plus d'une fois :", dup_invoices)

Factures payées plus d'une fois : 501


In [33]:
churn_labels = pd.read_parquet("../../data/processed/churn_labels.parquet")
churn_labels = churn_labels.rename(columns={"Customer ID": "customer_id"})
print(churn_labels.shape)
print(churn_labels.columns.tolist())

(4535, 4)
['customer_id', 'Churned', 'NbCommandesObs', 'Fidele']


In [34]:
cutoff = pd.Timestamp('2011-06-10')
real_customer_ids = set(transactions["Customer ID"].unique())

# --- Support : nb tickets + score de satisfaction moyen, avant coupure ---
support_obs = support[support["opened_date"] < cutoff]
support_agg = support_obs.groupby("customer_id").agg(
    nb_tickets=("ticket_id", "count"),
    avg_satisfaction=("satisfaction_score", "mean"),
)

# --- Fidélité : palier + solde de points, client réel uniquement (pas orphelin) ---
loyalty_obs = loyalty[
    (loyalty["enrollment_date"] < cutoff) & (loyalty["customer_id"].isin(real_customer_ids))
]
loyalty_agg = loyalty_obs.set_index("customer_id")[["tier", "points_balance"]]

# --- Marketing : préférences (pas de contrainte temporelle, attribut statique) ---
marketing_agg = marketing.set_index("customer_id")[["preferred_channel", "consent_email", "consent_sms"]]

# --- Retours : nb de retours avant coupure ---
returns_obs = returns[returns["return_date"] < cutoff]
returns_agg = returns_obs.groupby("customer_id").size().rename("nb_returns")

# --- Campagnes : déjà exclu les orphelins, ever_opened/converted ---
sends_obs = sends[(sends["sent_date"] < cutoff) & (sends["customer_id"].isin(real_customer_ids))].copy()
sends_obs["opened"] = sends_obs["opened"].fillna(False)
sends_obs["converted"] = sends_obs["converted"].fillna(False)
campaign_agg = sends_obs.groupby("customer_id").agg(
    nb_sends=("send_id", "count"), ever_opened=("opened", "any"), ever_converted=("converted", "any")
)

# --- Assemblage sur la population fidèles (churn_labels) ---
churn_labels = churn_labels.rename(columns={"Customer ID": "customer_id"})
print(churn_labels.columns.tolist())
enriched = churn_labels.merge(support_agg, on="customer_id", how="left")
enriched = enriched.merge(loyalty_agg, on="customer_id", how="left")
enriched = enriched.merge(marketing_agg, on="customer_id", how="left")
enriched = enriched.merge(returns_agg, on="customer_id", how="left")
enriched = enriched.merge(campaign_agg, on="customer_id", how="left")

enriched["has_support_ticket"] = enriched["nb_tickets"].notna()
enriched["is_loyalty_member"] = enriched["tier"].notna()
enriched["has_returns"] = enriched["nb_returns"].notna()
enriched["was_targeted"] = enriched["nb_sends"].notna()
enriched["ever_opened"] = enriched["ever_opened"].fillna(False)
enriched["low_satisfaction"] = enriched["avg_satisfaction"] < 3   # score bas = 1 ou 2 sur 5

loyal = enriched[enriched["Fidele"]]
print(loyal.shape)

['customer_id', 'Churned', 'NbCommandesObs', 'Fidele']
(2349, 20)


In [35]:
tests = ["has_support_ticket", "low_satisfaction", "is_loyalty_member", "has_returns", "was_targeted", "ever_opened"]

for col in tests:
    print(f"--- {col} ---")
    print(loyal.groupby(col)["Churned"].agg(["mean", "count"]))
    print()

print("--- tier (fidélité) ---")
print(loyal.groupby("tier")["Churned"].agg(["mean", "count"]))

print("--- preferred_channel ---")
print(loyal.groupby("preferred_channel")["Churned"].agg(["mean", "count"]))

--- has_support_ticket ---
                        mean  count
has_support_ticket                 
False               0.282979   1880
True                0.319829    469

--- low_satisfaction ---
                      mean  count
low_satisfaction                 
False             0.271906   2214
True              0.592593    135

--- is_loyalty_member ---
                       mean  count
is_loyalty_member                 
False              0.285283   1325
True               0.296875   1024

--- has_returns ---
                 mean  count
has_returns                 
False        0.481752    137
True         0.278481   2212

--- was_targeted ---
                  mean  count
was_targeted                 
False         0.222222     54
True          0.291939   2295

--- ever_opened ---
                 mean  count
ever_opened                 
False        0.280488   1476
True         0.306987    873

--- tier (fidélité) ---
            mean  count
tier                   
Bronze  0.2